In [368]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [369]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
d_model = 512
max_seq_len = 5000


In [370]:
class PositionalEncoding(nn.Module):

    def __init__(self, d_model, max_seq_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_seq_len, d_model)
        positions = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (- math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(positions * div_term)
        pe[:, 1::2] = torch.cos(positions * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [371]:
emb = torch.randn((32, 100, 512))
pe = PositionalEncoding(d_model=512, max_seq_len=5000)
x = pe(emb)

In [372]:
x.size()

torch.Size([32, 100, 512])

In [373]:
class TransformerEmbedding(nn.Module):

    def __init__(self, vocab_size, max_seq_len, d_model, dropout=0.1, positional_encoding=True):
        super().__init__()
        self.d_model = d_model
        self.positional_encoding = PositionalEncoding(d_model=d_model, max_seq_len=max_seq_len, dropout=dropout) if positional_encoding else None
        self.token_embedding = nn.Embedding(vocab_size, d_model)

    def forward(self, x):
        x = self.token_embedding(x)
        if self.positional_encoding is not None:
            x = math.sqrt(self.d_model) * x
            x = self.positional_encoding(x)
        return x

In [374]:
emb = TransformerEmbedding(20_000, 5000, 512)
sample = torch.randint(0, 20_000, (32, 100))
out = emb(sample)
out.size()

torch.Size([32, 100, 512])

In [375]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.d_model = d_model

        self.linear_key = nn.Linear(d_model, d_model)
        self.linear_query = nn.Linear(d_model, d_model)
        self.linear_value = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
        self.concat_linear = nn.Linear(d_model, d_model)

    def forward(self, query, key, value, mask):
        """
        query, key, value: (batch_size, seq_len, d_model)
        masks: (*, 1, 1, seq_len)
        """
        query = self.linear_query(query)
        key = self.linear_key(key)
        value = self.linear_value(value)

        # split into heads
        query = query.view(query.size(0), -1, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        key = key.view(key.size(0), -1, self.num_heads, self.d_k).permute(0, 2, 1, 3)
        value = value.view(value.size(0), -1, self.num_heads, self.d_k).permute(0, 2, 1, 3)

        scores = torch.matmul(query, key.permute(0, 1, 3, 2)) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e-9)
        scores = F.softmax(scores, dim=-1)
        scores = self.dropout(scores)
        attention = torch.matmul(scores, value)

        concat = attention.permute(0, 2, 1, 3).contiguous().view(attention.size(0), -1, self.d_model)
        out = self.concat_linear(concat)
        out = self.dropout(out)

        return out

In [376]:
sample_input = torch.randn((32, 100, 512))
mask = torch.ones((32, 1, 1, 100))
mha = MultiHeadAttention(d_model=512, num_heads=8)
out = mha(sample_input, sample_input, sample_input, mask)
out.size()

torch.Size([32, 100, 512])

In [377]:
class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [ ]:
class EncoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout = nn.Dropout(dropout)
        self.multi_head = MultiHeadAttention(d_model, num_heads, dropout)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask):
        out = self.multi_head(x, x, x, mask)
        out = self.layer_norm1(out + x)

        feed_forward = self.feed_forward(out)
        feed_forward = self.dropout(feed_forward)

        out = self.layer_norm2(out + feed_forward)
        return out
    

# pre LN

class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        
        # Sub-layers
        self.multi_head = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # Layer Normalization
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        norm_x = self.layer_norm1(x)
        attn_out = self.multi_head(norm_x, norm_x, norm_x, mask)
        x = x + self.dropout(attn_out) 
        
        norm_x2 = self.layer_norm2(x)
        ff_out = self.feed_forward(norm_x2)
        x = x + self.dropout(ff_out)
        
        return x

In [379]:
class DecoderLayer(nn.Module):

    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.dropout = nn.Dropout(dropout)
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.layer_norm2 = nn.LayerNorm(d_model)
        self.layer_norm3 = nn.LayerNorm(d_model)
        self.multi_head1 = MultiHeadAttention(d_model, num_heads, dropout)
        self.multi_head2 = MultiHeadAttention(d_model, num_heads, dropout)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, encoded_rep, casual_mask, encoder_mask):
        out = self.multi_head1(x, x, x, casual_mask)
        out = self.layer_norm1(out + x)

        cross_out = self.multi_head2(out, encoded_rep, encoded_rep, encoder_mask)
        out = self.layer_norm2(out + cross_out)

        feed_forward = self.feed_forward(out)
        feed_forward = self.dropout(feed_forward)
        out = self.layer_norm3(out + feed_forward)

        return out


In [380]:
from torchinfo import summary

In [381]:
encoder_layer = EncoderLayer(d_model, num_heads=8, d_ff=2048, dropout=0.1)
summary(encoder_layer)

Layer (type:depth-idx)                   Param #
EncoderLayer                             --
├─MultiHeadAttention: 1-1                --
│    └─Linear: 2-1                       262,656
│    └─Linear: 2-2                       262,656
│    └─Linear: 2-3                       262,656
│    └─Dropout: 2-4                      --
│    └─Linear: 2-5                       262,656
├─FeedForward: 1-2                       --
│    └─Linear: 2-6                       1,050,624
│    └─Linear: 2-7                       1,049,088
│    └─Dropout: 2-8                      --
├─LayerNorm: 1-3                         1,024
├─LayerNorm: 1-4                         1,024
├─Dropout: 1-5                           --
Total params: 3,152,384
Trainable params: 3,152,384
Non-trainable params: 0

In [382]:
import pandas as pd
import numpy as np

train_dataframe = pd.read_csv("./data/text_classification/news/train.csv")
train_dataframe.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [383]:
# get max word length
max_word_length = train_dataframe['Description'].str.split().str.len().max()
max_word_length

173

In [ ]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel 
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

class BPETokenizer:
    def __init__(self, vocab_size=15_000, min_frequency=2):
        """
        High-performance BPE Tokenizer that mirrors GPT's byte-level space tracking.
        """
        self.vocab_size = vocab_size
        self.min_frequency = min_frequency
        
        self.special_tokens = ["[PAD]", "[CLS]"]
        
        self.tokenizer = Tokenizer(BPE())
        
        self.tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)
        
        self.tokenizer.decoder = ByteLevelDecoder()

    def train(self, text_series):
        """Trains the tokenizer directly on your text classification column."""
        print(f"Training GPT-style BPE Tokenizer...")
        
        clean_corpus = [text for text in text_series if isinstance(text, str)]
        
        trainer = BpeTrainer(
            vocab_size=self.vocab_size,
            min_frequency=self.min_frequency,
            special_tokens=self.special_tokens,
            show_progress=True 
        )
        
        self.tokenizer.train_from_iterator(clean_corpus, trainer)
        print(f"Training completed! Final Vocab Size: {self.tokenizer.get_vocab_size()}")

    def encode(self, text: str) -> list:
        if not isinstance(text, str):
            return []
        return self.tokenizer.encode(text).ids

    def decode(self, ids: list) -> str:
        return self.tokenizer.decode(ids)

    def save_to_json(self, path: str):
        directory = os.path.dirname(path)
        if directory and not os.path.exists(directory):
            os.makedirs(directory)
        self.tokenizer.save(path)
        print(f"Tokenizer checkpoint saved to: {path}")

    @classmethod
    def load_from_json(cls, path: str):
        instance = cls()
        instance.tokenizer = Tokenizer.from_file(path)
        return instance

In [385]:
tokenizer = BPETokenizer(vocab_size=10_000)
tokenizer.train(train_dataframe['Description'].dropna())

Training GPT-style BPE Tokenizer...
Training completed! Final Vocab Size: 10000


In [386]:
tokenizer.save_to_json("./tokenizer_checkpoints/gpt_style_bpe_tokenizer.json")

Tokenizer checkpoint saved to: ./tokenizer_checkpoints/gpt_style_bpe_tokenizer.json


In [387]:
tokenizer.load_from_json("./tokenizer_checkpoints/gpt_style_bpe_tokenizer.json")

In [388]:
tokenizer.encode("What is the capital of France?")

[4573, 211, 92, 1729, 112, 1965, 28]

In [389]:
tokenizer.decode([258, 232, 204, 3169, 239, 3546, 31])

' plctri light de fixC'

In [390]:
vocab_size = tokenizer.vocab_size
n_layers = 4
dropout = 0.2
d_model = 512
max_seq_len = 100
d_ff = 2048
num_heads = 8
batch_size = 64
classes = [
    'World',
    'Sports',
    'Business',
    'Science & Technology',
]

In [391]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, max_seq_len):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = self.dataframe.iloc[idx]['Title']
        label = self.dataframe.iloc[idx]['Class Index'] - 1
        
        # Tokenize the text
        token_ids = self.tokenizer.encode(text)
        # Get the [CLS] token
        cls_token_id = self.tokenizer.encode("[CLS]")
        # Add [CLS] token at the beginning
        token_ids = cls_token_id + token_ids
        # Truncate to max_seq_len (now includes [CLS])
        token_ids = token_ids[:self.max_seq_len]
        token_ids = torch.tensor(token_ids)
        return token_ids, torch.tensor(label)

In [392]:
train_dataset = TextDataset(train_dataframe, tokenizer, max_seq_len)
train_dataset[0]

(tensor([   1, 1680,  291,   13, 6414, 1506,  521, 8280,  208, 1127,   92, 2909,
          196,  355,    9]),
 tensor(2))

In [393]:
def collate_fn(batch):
    token_ids, labels = zip(*batch)
    # Pad sequences to the maximum length in the batch
    max_length = max(len(ids) for ids in token_ids)
    padded_token_ids = torch.zeros(len(token_ids), max_length, dtype=torch.long)
    padded_masks = torch.zeros(len(token_ids), max_length, dtype=torch.bool)
    
    for i, ids in enumerate(token_ids):
        padded_token_ids[i, :len(ids)] = ids
        padded_masks[i, :len(ids)] = True
    
    return padded_token_ids, padded_masks.unsqueeze(1).unsqueeze(2), torch.tensor(labels)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)

In [394]:
next(iter(train_loader))[0].size()

torch.Size([64, 29])

In [395]:
val_dataframe = pd.read_csv("./data/text_classification/news/test.csv")
val_dataset = TextDataset(val_dataframe, tokenizer, max_seq_len)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

In [396]:
class TransformerEncoder(nn.Module):

    def __init__(self):
        super().__init__()
        self.embedding = TransformerEmbedding(vocab_size, max_seq_len, d_model, dropout=dropout, positional_encoding=False)
        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads=num_heads, d_ff=d_ff, dropout=dropout) for _ in range(n_layers)])
        self.final_layer_norm = nn.LayerNorm(d_model)
        self.logits = nn.Linear(d_model, len(classes))

    def forward(self, x, mask):
        encoded_rep = self.embedding(
            x
        )
        for layer in self.encoder_layers:
            encoded_rep = layer(encoded_rep, mask)

        encoded_rep = self.final_layer_norm(encoded_rep)
        
        return self.logits(encoded_rep[:, 0, :])

In [397]:
sample_encoder = TransformerEncoder()
sample_inp, sample_mask, sample_label = next(iter(train_loader))
sample_out = sample_encoder(sample_inp, sample_mask)
sample_out.size()

torch.Size([64, 4])

In [398]:
model = TransformerEncoder().to(device)

In [ ]:
import torch
from torch.optim.lr_scheduler import LambdaLR

optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=1.0, 
)

def get_lr_multiplier(step, warmup_steps=4000, d_model=d_model):
    step = max(1, step) 
    
    scale = (d_model ** -0.5) * min(step ** (-0.5), step * (warmup_steps ** (-1.5)))
    
    return scale

scheduler = LambdaLR(optimizer, lr_lambda=get_lr_multiplier)

In [400]:
criterion = nn.CrossEntropyLoss()

In [401]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/ag_news_classification_experiment_2")

dummy_input = torch.randint(0, vocab_size, (1, len(classes))).to(device)
writer.add_graph(model, (dummy_input, torch.ones((1, 1, 1, len(classes))).to(device)), verbose=True)

graph(%self.1 : __torch__.___torch_mangle_1006.TransformerEncoder,
      %x.1 : Long(1, 4, strides=[4, 1], requires_grad=0, device=cuda:0),
      %mask : Float(1, 1, 1, 4, strides=[4, 4, 4, 1], requires_grad=0, device=cuda:0)):
  %logits : __torch__.torch.nn.modules.linear.___torch_mangle_1005.Linear = prim::GetAttr[name="logits"](%self.1)
  %final_layer_norm : __torch__.torch.nn.modules.normalization.___torch_mangle_1004.LayerNorm = prim::GetAttr[name="final_layer_norm"](%self.1)
  %encoder_layers : __torch__.torch.nn.modules.container.___torch_mangle_1003.ModuleList = prim::GetAttr[name="encoder_layers"](%self.1)
  %_3 : __torch__.___torch_mangle_1002.EncoderLayer = prim::GetAttr[name="3"](%encoder_layers)
  %encoder_layers.5 : __torch__.torch.nn.modules.container.___torch_mangle_1003.ModuleList = prim::GetAttr[name="encoder_layers"](%self.1)
  %_2 : __torch__.___torch_mangle_988.EncoderLayer = prim::GetAttr[name="2"](%encoder_layers.5)
  %encoder_layers.3 : __torch__.torch.nn.module

In [ ]:
import torch
from tqdm import tqdm

epochs = 100
global_step = 0
print_interval = 1

print(f"Starting training loop for {epochs} epochs...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    show_progress = (epoch + 1) % print_interval == 0 or epoch == 0
    
    train_bar = tqdm(
        train_loader, 
        desc=f"Epoch {epoch+1}/{epochs} [Train]", 
        disable=not show_progress,
        leave=False
    )
    
    for batch_x, mask, batch_y in train_bar:
        batch_x = batch_x.to(device)
        mask = mask.to(device) if mask is not None else None
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        outputs = model(batch_x, mask)
        loss = criterion(outputs, batch_y)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.1)
        
        optimizer.step()
        
        scheduler.step() 
        
        current_loss = loss.item()
        train_loss += current_loss
        
        writer.add_scalar("Loss/Train_Step", current_loss, global_step)
        
        current_lr = optimizer.param_groups[0]['lr']
        writer.add_scalar("Params/Learning_Rate", current_lr, global_step)
        
        global_step += 1
        train_bar.set_postfix(loss=f"{current_loss:.4f}")
        
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    val_bar = tqdm(
        val_loader, 
        desc=f"Epoch {epoch+1}/{epochs} [Val]", 
        disable=not show_progress,
        leave=False
    )
    
    with torch.no_grad():
        for batch_x, mask, batch_y in val_bar:
            batch_x = batch_x.to(device)
            mask = mask.to(device) if mask is not None else None
            batch_y = batch_y.to(device)
            
            outputs = model(batch_x, mask)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
            
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == batch_y).sum().item()
            total += batch_y.size(0)
            
    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total if total > 0 else 0.0
    
    writer.add_scalar("Loss/Epoch_Train", avg_train_loss, epoch)
    writer.add_scalar("Loss/Epoch_Val", avg_val_loss, epoch)
    writer.add_scalar("Accuracy/Val", val_accuracy, epoch)
    
    if (epoch + 1) % print_interval == 0 or epoch == 0:
        print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_accuracy:.4f}")

writer.close()
print("Training complete! High-resolution history saved to TensorBoard.")

Starting training loop for 100 epochs...


Epoch 001/100 | Train Loss: 0.8928 | Val Loss: 0.6440 | Val Acc: 0.7620


Epoch 002/100 | Train Loss: 0.6214 | Val Loss: 0.6215 | Val Acc: 0.7703


Epoch 003/100 | Train Loss: 0.5809 | Val Loss: 0.5872 | Val Acc: 0.7929


Epoch 004/100 | Train Loss: 0.5057 | Val Loss: 0.5879 | Val Acc: 0.7987


...

Epoch 010/100 | Train Loss: 0.3398 | Val Loss: 0.5677 | Val Acc: 0.8262


Epoch 011/100 | Train Loss: 0.3270 | Val Loss: 0.5798 | Val Acc: 0.8251


Epoch 012/100 | Train Loss: 0.3136 | Val Loss: 0.6166 | Val Acc: 0.8230


Epoch 013/100 | Train Loss: 0.3021 | Val Loss: 0.5939 | Val Acc: 0.8276


KeyboardInterrupt: 